##Creating Flag Parameter

In [0]:
dbutils.widgets.text('incremental_flag','0')

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')
print(type(incremental_flag))

##Creating Dimension Models

###Fetch Relative columns

In [0]:
df_src = spark.sql('''
SELECT DISTINCT(Branch_ID) AS Branch_ID, BranchName FROM parquet.`abfss://carsproject@pravdatalake.dfs.core.windows.net/silver`
''')

In [0]:
df_src.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### dim model sink initial and increment(just bring the schema if table NOT EXISTS)

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_branch'):
    df_sink = spark.sql('''
        SELECT dim_branch_key, Branch_ID, BranchName
        FROM cars_catalog.gold.dim_branch
        ''')    

else:

    df_sink = spark.sql('''
        SELECT 1 as dim_branch_key, Branch_ID, BranchName
        FROM parquet.`abfss://carsproject@pravdatalake.dfs.core.windows.net/silver`
        WHERE 1=0 
        ''')


In [0]:
df_sink.display()

### Brininging already existing records

In [0]:
df_filter = df_src.join(df_sink, df_src['Branch_ID']==df_sink['Branch_ID'], 'left').select(df_src['Branch_ID'], df_src['BranchName'], df_sink['dim_branch_key'])

In [0]:
df_filter.display()

##df_filter_old

In [0]:
df_filter_old = df_filter.filter(col('dim_branch_key').isNotNull())

###df_filter_new

In [0]:
df_filter_new = df_filter.filter(col('dim_branch_key').isNull()).select(df_src['Branch_ID'], df_src['BranchName']) 

##Create Surrogate key

**Fetch the max Surrogate key from existing table**

In [0]:
if (incremental_flag == '0'):
    max_value = 0
else:
    max_value_df = spark.sql("select max(dim_branch_key) from cars_catalog.gold.dim_branch")
    max_value = max_value_df.collect()[0][0]

**Create Surrogate key column and ADD the max surrogate key**

In [0]:
df_filter_new = df_filter_new.withColumn('dim_branch_key', max_value+monotonically_increasing_id())

In [0]:
df_filter_new.display()

###Create Final DF - df_filter_old + df_filter_new

In [0]:
df_final = df_filter_new.union(df_filter_old)

###SCD Type - 1(UPSERT)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_branch'):
    delta_tbl = DeltaTable.forPath(spark,"abfss://carsproject@pravdatalake.dfs.core.windows.net/dim_branch")
    delta_tbl.alias("trg").merge(df_final.alias("src"),"trg.dim_branch_key = src.dim_branch_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
        
#Initial run
else:
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option('path',"abfss://carsproject@pravdatalake.dfs.core.windows.net/dim_branch")\
        .saveAsTable("cars_catalog.gold.dim_branch")

In [0]:
%sql
SELECT * FROM cars_catalog.gold.dim_branch;